# Silver 2: retry the failed parses

Task `silver_retry` of the job `ingest_financebench`. Takes the PDFs whose parse failed in a **retryable** way (rate limit, internal error) and has attempts left, and parses them again with the **same function** as the first parse (`parse_docs`). There is no special repair code: a PDF that now parses completely goes to `silver_parsed`, and one that fails again gets another row in `silver_log` with the next attempt number.

Every retryable PDF is retried in one run. To go easy on a limited quota (the Databricks free tier):

| Guard | What it does |
| --- | --- |
| Cheapest first | The shortest PDFs go first, so the most documents are fixed early |
| One PDF at a time, with a pause | `retry_pause_seconds`, so the rate limit is not hit again |
| Stop when the service is limiting | Two failures in a row stop the retries for this run |
| Attempt cap | After `max_attempts` a PDF is left alone. It stays in `silver_failures` |
| Never retried | Failures that cannot succeed (a PDF over 500 pages, a failed conversion) |

A retry parses the whole PDF again. Retrying only the failed pages would be cheaper in parse pages, but it needs the PDF cut and the result merged back, which was most of the complexity of an earlier version.

In [ ]:
import time

from tables import Names, create_tables
from parsing import parse_docs

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("max_attempts", "3")
dbutils.widgets.text("retry_pause_seconds", "10")

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"))
max_attempts = int(dbutils.widgets.get("max_attempts"))
retry_pause_seconds = int(dbutils.widgets.get("retry_pause_seconds"))

create_tables(spark, n)

In [ ]:
candidates = spark.sql(f"""
SELECT path, pdf_pages FROM {n.failures}
WHERE stage = 'parse' AND retryable AND attempts < {max_attempts}
ORDER BY pdf_pages
""").collect()
print(f"{len(candidates)} PDFs have a retryable parse failure.")

spent, in_a_row = 0, 0
for c in candidates:
    if in_a_row >= 2:  # the service keeps failing: stop, run again later
        break
    good, bad = parse_docs(spark, n, spark.table(n.bronze).filter(f"path = '{c.path}'"))
    spent += c.pdf_pages
    in_a_row = in_a_row + 1 if bad else 0
    print(f"{c.path.split('/')[-1]}: {'parsed' if good else 'failed again'} ({c.pdf_pages} pages)")
    time.sleep(retry_pause_seconds)
print(f"Retry pages spent: {spent}." + (" Stopped: the service keeps failing." if in_a_row >= 2 else ""))